# PISA 2025 extraction (low-memory, resumable)
Runtime → Run all. If it stops, just Run all again: finished parts are skipped. Output: Drive/PISA2025_extract.

In [ ]:
!pip -q install pyreadstat
import os, re, glob, time, shutil, subprocess, gc, pyreadstat, pandas as pd, numpy as np
from google.colab import drive
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/PISA2025_extract'; os.makedirs(OUT,exist_ok=True)
W='/content/w'; P='/content/parts'; os.makedirs(W,exist_ok=True)
BASE='https://webfs.oecd.org/pisa2022/2025/'
ITEM=re.compile(r'^[A-Z]{3,}\d{3}Q\d{2}')
STU_KEEP=('CNT','CNTSCHID','CNTSTUID','BOOKID','ADMINMODE','ST004D01T','ESCS','AIUSESCH','EFFORT1','EFFORT2','W_FSTUWT')
STU_PREF=('ST438','ST436','ST331','W_FSTURWT','PV')

def extract(z,tag,colsel):
    if os.path.exists(f'{OUT}/{tag}.done'): print(tag,'already done'); return
    t=time.time(); shutil.rmtree(P,ignore_errors=True); os.makedirs(P)
    for f in glob.glob(W+'/*'): os.remove(f)
    subprocess.run(['wget','-q','-O',f'{W}/{z}',BASE+z],check=True)
    subprocess.run(['unzip','-o','-q',f'{W}/{z}','-d',W],check=True); os.remove(f'{W}/{z}')
    f=glob.glob(W+'/*.sav')[0]; print(tag,'unzipped',int(time.time()-t),'s',flush=True)
    _,m=pyreadstat.read_sav(f,metadataonly=True); cols=[c for c in m.column_names if colsel(c)]
    for i,(d,_) in enumerate(pyreadstat.read_file_in_chunks(pyreadstat.read_sav,f,chunksize=20000,usecols=cols)):
        num=d.select_dtypes('number').columns; d[num]=d[num].astype('float32')
        for c,g in d.groupby('CNT'): g.to_parquet(f'{P}/{c}__{i:04d}.parquet',index=False)
        del d; gc.collect()
        if i%5==0: print(tag,'chunk',i,int(time.time()-t),'s',flush=True)
    os.remove(f)
    for c in sorted({os.path.basename(p).split('__')[0] for p in glob.glob(P+'/*.parquet')}):
        ps=sorted(glob.glob(f'{P}/{c}__*.parquet'))
        g=pd.concat([pd.read_parquet(p) for p in ps]).dropna(axis=1,how='all')
        g.to_parquet(f'{OUT}/{tag}_{c}.parquet',index=False); del g; gc.collect()
    shutil.rmtree(P,ignore_errors=True); open(f'{OUT}/{tag}.done','w').close(); print(tag,'DONE',int(time.time()-t),'s',flush=True)

extract('CY09_MS_STU_PUF.zip','STU',lambda c: c in STU_KEEP or (c.startswith(STU_PREF) and (not c.startswith('PV') or c.endswith(('MATH','READ','SCIE')))))
extract('CY09_MS_COG_PROCESS_PUF.zip','PROC',lambda c: not ITEM.match(c) or c.endswith(('TT','A','F')))
extract('CY09_MS_COG_PUF.zip','COG',lambda c: not ITEM.match(c) or c.endswith('S'))

S={}
for p in sorted(glob.glob(OUT+'/STU_*.parquet')):
    c=p.split('_')[-1][:-8]; g=pd.read_parquet(p,columns=None)
    S[c]={'n':len(g),'ai_cov':g['AIUSESCH'].notna().mean() if 'AIUSESCH' in g else 0,'st438_cov':g['ST438Q01DA'].notna().mean() if 'ST438Q01DA' in g else 0}
for p in sorted(glob.glob(OUT+'/PROC_*.parquet')):
    c=p.split('_')[-1][:-8]; g=pd.read_parquet(p); tt=[x for x in g if ITEM.match(x) and x.endswith('TT')]
    S.setdefault(c,{}).update({'n_proc':len(g),'items_tt':len(tt),'tt_per_student':float(g[tt].notna().sum(1).mean()) if tt else 0,
        'med_tt':float(np.nanmedian(g[tt].values)) if tt else np.nan,'math_items':sum(x.startswith('CM') for x in tt),
        'read_items':sum(x.startswith('CR') for x in tt),'sci_items':sum(x.startswith(('CS','CAC')) for x in tt)}); del g; gc.collect()
for p in sorted(glob.glob(OUT+'/COG_*.parquet')):
    c=p.split('_')[-1][:-8]; cols=pd.read_parquet(p).columns
    S.setdefault(c,{}).update({'items_s':sum(bool(ITEM.match(x)) for x in cols)})
sm=pd.DataFrame(S).T.sort_index(); sm.to_csv(f'{OUT}/summary.csv'); print(sm.loc[['ARM']].T); print('ALL DONE')
